# 01 — Data Pipeline
## Brain Tumour Detection — Final Project
=====================================

Topics covered:
  1.  Why the reference notebooks' numbers do not transfer
  2.  The dataset as shipped — what is actually in the folders
  3.  The pre-augmented duplicates, and why they leave the test set
  4.  Decoding once — the in-RAM cache
  5.  Cropping to content — removing a nuisance variable
  6.  The split — official test set, stratified validation
  7.  Normalisation statistics from the training split only
  8.  Augmentation for MRI — what is anatomically defensible
  9.  Looking at the data before trusting it
  10. Summary and verification checks

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src import config, data, viz
from src.config import CACHE_DIR, FACE, PALETTE, TEST_DIR, TRAIN_DIR

In [2]:
# 1. WHY THE REFERENCE NOTEBOOKS' NUMBERS DO NOT TRANSFER
"""
Days 1-8 built every component of this model against 1000 images produced by
make_synthetic_mri(). That generator draws one stereotyped cue per class: a
small bright dot near the bottom is always pituitary, scattered ellipses are
always glioma. The classes are separable by a rule a human could write down in
an afternoon, which is why Day 7 reached 100% test accuracy with a NEGATIVE
generalisation gap.

That result was never evidence about tumour classification. It was evidence
that the training loop, the loss and the checkpointing were wired correctly —
which is exactly what a reference notebook should establish, and no more.

Everything from here runs on real scans, so two things change. Every accuracy
figure becomes a claim about the problem rather than about the plumbing, and
every preprocessing constant computed on synthetic data has to be recomputed.
"""
print("Reference pipeline (Days 1-8):  1000 synthetic images, 4 stereotyped cues")
print("This pipeline:                  7200 real MRI scans, Kaggle Nickparvar v2")
print()
print("Constants that must NOT be carried over:")
print("  MEAN = 0.4382, STD = 0.2220   <- computed on synthetic data, recomputed in section 7")

Reference pipeline (Days 1-8):  1000 synthetic images, 4 stereotyped cues
This pipeline:                  7200 real MRI scans, Kaggle Nickparvar v2

Constants that must NOT be carried over:
  MEAN = 0.4382, STD = 0.2220   <- computed on synthetic data, recomputed in section 7


In [3]:
# 2. THE DATASET AS SHIPPED
"""
The download arrives pre-split into Training/ and Testing/, each holding one
folder per class. We keep that split rather than pooling and re-splitting: the
published results other people report on this dataset use the official test
set, and re-splitting would quietly make our number incomparable to theirs.

Note the folder is spelled `notumor`. The reference notebooks used `notumour`.
Alphabetical ordering is unaffected so the label indices are identical either
way, but a hardcoded class list copied from Day 7 would silently mislabel every
axis of the confusion matrix. Class names are therefore always read from disk.
"""
for split_dir in (TRAIN_DIR, TEST_DIR):
    counts = {d.name: len(list(d.iterdir())) for d in sorted(split_dir.iterdir())}
    print(f"{split_dir.name:<9} {counts}   total {sum(counts.values())}")

modes, sizes = {}, []
for cls_dir in sorted(TRAIN_DIR.iterdir()):
    for p in list(cls_dir.iterdir())[:40]:
        im = Image.open(p)
        modes[im.mode] = modes.get(im.mode, 0) + 1
        sizes.append(im.size)

print(f"\nimage modes in a 160-file sample: {modes}")
print(f"image sizes: min {min(sizes)}  max {max(sizes)}")
print("\n-> the data genuinely mixes single-channel L and 3-channel RGB files,")
print("   so grayscale conversion is mandatory, not cosmetic: ConvBlock(1, 32)")
print("   would raise a shape error the moment an RGB scan arrived.")

Training  {'glioma': 1400, 'meningioma': 1400, 'notumor': 1400, 'pituitary': 1400}   total 5600
Testing   {'glioma': 400, 'meningioma': 400, 'notumor': 400, 'pituitary': 400}   total 1600

image modes in a 160-file sample: {'L': 58, 'RGB': 102}
image sizes: min (192, 192)  max (900, 741)

-> the data genuinely mixes single-channel L and 3-channel RGB files,
   so grayscale conversion is mandatory, not cosmetic: ConvBlock(1, 32)
   would raise a shape error the moment an RGB scan arrived.


In [4]:
# 3. THE PRE-AUGMENTED DUPLICATES
"""
The class counts are suspiciously round — exactly 1400 and 400 everywhere. They
are round because the uploader padded them. Files whose names contain '-aug-'
are transformed copies of other images in the same dataset, and only the
meningioma class needed them.

An augmented copy in the TRAINING set is harmless; it is free augmentation.
An augmented copy in the TEST set is not. If its source image sits in Training,
the model is being scored on a rotated version of something it has already
memorised, and the meningioma recall we report would be inflated by exactly the
kind of leakage this project is supposed to be careful about.

So: drop the 103 in Testing, keep the 100 in Training. That asymmetry is
deliberate and gets stated in the report.
"""
for split_dir in (TRAIN_DIR, TEST_DIR):
    print(f"{split_dir.name}:")
    for cls_dir in sorted(split_dir.iterdir()):
        files = list(cls_dir.iterdir())
        aug = [f for f in files if '-aug-' in f.name.lower()]
        flag = "  <- padded" if aug else ""
        print(f"  {cls_dir.name:<12} {len(files):>5} files   {len(aug):>4} augmented{flag}")

Training:
  glioma        1400 files      0 augmented
  meningioma    1400 files    100 augmented  <- padded
  notumor       1400 files      0 augmented
  pituitary     1400 files      0 augmented
Testing:
  glioma         400 files      0 augmented
  meningioma     400 files    103 augmented  <- padded
  notumor        400 files      0 augmented
  pituitary      400 files      0 augmented


In [5]:
# 4. DECODING ONCE — THE IN-RAM CACHE
"""
The reference notebooks read every image from disk on every access, through
ImageFolder. At 1000 small synthetic PNGs that costs nothing. Here it is the
single dominant cost: 7,200 JPEGs at up to 1000px, and num_workers must stay 0
because Windows re-imports the worker process rather than forking it.

Since decoding produces the same pixels every epoch, it only needs to happen
once. build_cache() decodes each split into a uint8 array held in RAM;
augmentation still runs per access, because that is the part that is supposed
to differ every epoch.

The cache is stored at 224px rather than the 128px we train at, so that
notebook 04 can ablate resolution without rebuilding it.
"""
t0 = time.time()
if (CACHE_DIR / "train_img.npy").exists():
    train_img = np.load(CACHE_DIR / "train_img.npy")
    train_lab = np.load(CACHE_DIR / "train_lab.npy")
    test_img  = np.load(CACHE_DIR / "test_img.npy")
    test_lab  = np.load(CACHE_DIR / "test_lab.npy")
    CLASSES   = sorted(d.name for d in TRAIN_DIR.iterdir())
    print(f"loaded existing cache ({time.time()-t0:.1f}s)")
else:
    train_img, train_lab, CLASSES  = data.build_cache(TRAIN_DIR)
    test_img,  test_lab,  test_cls = data.build_cache(TEST_DIR, keep_augmented=False)
    assert CLASSES == test_cls, "Training/ and Testing/ disagree on class order"
    np.save(CACHE_DIR / "train_img.npy", train_img); np.save(CACHE_DIR / "train_lab.npy", train_lab)
    np.save(CACHE_DIR / "test_img.npy",  test_img);  np.save(CACHE_DIR / "test_lab.npy",  test_lab)
    print(f"built cache from JPEGs ({time.time()-t0:.0f}s)")

print(f"\ntrain cache {train_img.shape}  {train_img.nbytes/1e6:.0f} MB")
print(f"test  cache {test_img.shape}  {test_img.nbytes/1e6:.0f} MB")
print(f"classes: {CLASSES}")
assert len(test_lab) == 1497, f"expected 1600-103 = 1497 test images, got {len(test_lab)}"
print(f"\ntest set is {len(test_lab)} images = 1600 shipped - 103 augmented  [OK]")

loaded existing cache (0.1s)

train cache (5600, 224, 224)  281 MB
test  cache (1497, 224, 224)  75 MB
classes: ['glioma', 'meningioma', 'notumor', 'pituitary']

test set is 1497 images = 1600 shipped - 103 augmented  [OK]


In [6]:
# 5. CROPPING TO CONTENT
"""
Roughly 45% of a raw slice is black background, and crucially it is not a
consistent 45% — these scans come from different fields of view, so the brain
occupies a different fraction of each image.

Resizing without cropping therefore rescales the brain itself by an arbitrary
per-image factor, and the network has to spend capacity learning to ignore a
nuisance variable we can simply delete. Cropping to the bounding box of the
non-black pixels normalises scale before the model ever sees the image.

It has a second use. RandomAffine fills the corners it rotates into with a
constant, and fill=0 is only the right choice if everything outside the brain
really is black — which cropping guarantees.
"""
sample_path = sorted((TRAIN_DIR / CLASSES[0]).iterdir())[0]
raw = Image.open(sample_path).convert("L")
cropped = data.crop_to_content(raw)

fig, axes = viz.styled_fig(1, 3, figsize=(11, 4))
axes[0].imshow(raw, cmap='gray');     axes[0].set_title(f"raw {raw.size}", fontsize=9)
axes[1].imshow(cropped, cmap='gray'); axes[1].set_title(f"cropped {cropped.size}", fontsize=9)
axes[2].imshow(cropped.resize((128, 128)), cmap='gray')
axes[2].set_title("cropped + resized 128", fontsize=9)
for ax in axes: ax.axis('off')
plt.suptitle("Cropping removes a per-image scale difference", fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "preprocessing.png")

frac = np.asarray(raw).size and (np.asarray(raw) <= 10).mean()
print(f"\nbackground pixels in this scan: {frac:.1%}")
print(f"area kept after cropping:      {cropped.size[0]*cropped.size[1]/(raw.size[0]*raw.size[1]):.1%}")

  saved -> outputs/preprocessing.png

background pixels in this scan: 43.4%
area kept after cropping:      71.9%


In [7]:
# 6. THE SPLIT
"""
Testing/ is set aside untouched and is not looked at again until notebook 03.
Training/ is split 85:15 into train and validation, stratified so each class
keeps its proportions.

The validation set is what every decision in notebook 02 is made against —
which epoch to checkpoint, when to stop. That is precisely why it cannot also
be the set we report: a number chosen to be the maximum of many is biased
upward, and the test set exists to give an estimate nothing was selected on.

One honest limitation, stated here because it cannot be fixed: this dataset
ships no patient identifiers. Multiple slices from one patient are near
duplicates, and without IDs we cannot guarantee they did not land on both sides
of the split. Our test set is at least a split we did not draw ourselves.
"""
train_idx, val_idx = data.stratified_split(train_lab)
assert not (set(train_idx) & set(val_idx)), "train and val overlap"

print(f"train {len(train_idx):>5}   val {len(val_idx):>4}   test {len(test_lab):>5}")
print(f"overlap between train and val: {len(set(train_idx) & set(val_idx))}\n")
print(f"{'class':<12}{'train':>8}{'val':>7}{'test':>7}")
print("-" * 34)
for c, name in enumerate(CLASSES):
    print(f"{name:<12}{(train_lab[train_idx]==c).sum():>8}"
          f"{(train_lab[val_idx]==c).sum():>7}{(test_lab==c).sum():>7}")

train  4760   val  840   test  1497
overlap between train and val: 0

class          train    val   test
----------------------------------
glioma          1190    210    400
meningioma      1190    210    297
notumor         1190    210    400
pituitary       1190    210    400


In [8]:
# 7. NORMALISATION STATISTICS FROM THE TRAINING SPLIT ONLY
"""
Normalisation shifts and scales every pixel by two constants. Those constants
are learned from data, which makes them exactly as capable of leaking as model
weights: computing them over the whole dataset lets information about the test
scans reach the training pipeline.

They are computed after cropping and resizing, so that they describe the
tensors the model actually receives rather than some earlier version of them.

Compare the result against the synthetic values. The reference notebooks used
MEAN 0.4382 — carrying that constant across would have centred every real scan
around the wrong point.
"""
MEAN, STD = data.compute_stats(train_img, train_idx)
np.save(CACHE_DIR / "norm.npy", np.array([MEAN, STD]))

print(f"real data    MEAN = {MEAN:.4f}   STD = {STD:.4f}")
print(f"synthetic    MEAN = 0.4382   STD = 0.2220")
print(f"\ndifference in mean: {abs(MEAN-0.4382):.4f} — real scans are much darker,")
print("because roughly half of every cropped slice is still near-black background.")

real data    MEAN = 0.2250   STD = 0.1901
synthetic    MEAN = 0.4382   STD = 0.2220

difference in mean: 0.2132 — real scans are much darker,
because roughly half of every cropped slice is still near-black background.


In [9]:
# 8. AUGMENTATION FOR MRI
"""
Day 8 measured that augmentation was the single most effective regulariser
available. Which transforms are defensible on brain scans is a separate
question from whether augmentation helps, and it is the one a medical imaging
report gets challenged on.

RandomHorizontalFlip is dropped. The usual argument against it — that brains
are not symmetric — is weak, because axial and coronal slices are grossly
bilaterally symmetric and tumours genuinely occur on both sides. The real
problem is that this dataset mixes acquisition planes. A mirrored SAGITTAL
slice is anatomically impossible: it puts the frontal lobe where the cerebellum
belongs. Mixed planes make the flip unsafe, so it goes.

RandomAffine stays. Small rotations, shifts and scalings correspond to real
variation in how a patient was positioned in the scanner.

ColorJitter stays, mildly — Day 8 dismissed it too quickly. MRI intensity is
not calibrated in absolute physical units the way CT Hounsfield numbers are.
Scanner, sequence and display windowing all shift brightness and contrast, so
jittering them reproduces genuine acquisition variance rather than inventing a
distortion that never happens.
"""
train_tf = data.make_transforms(MEAN, STD, augment=True)
eval_tf  = data.make_transforms(MEAN, STD, augment=False)

print("train:", *[f"  {t.__class__.__name__}" for t in train_tf.transforms], sep="\n")
print("\neval: ", *[f"  {t.__class__.__name__}" for t in eval_tf.transforms],  sep="\n")

src_img = Image.fromarray(train_img[train_idx[0]], mode="L")
fig, axes = viz.styled_fig(1, 6, figsize=(13, 2.6))
axes[0].imshow(src_img, cmap='gray'); axes[0].set_title("original", fontsize=8)
for ax in axes[1:]:
    ax.imshow(viz.denorm(train_tf(src_img), MEAN, STD), cmap='gray')
    ax.set_title("augmented", fontsize=8)
for ax in axes: ax.axis('off')
plt.suptitle("One scan, five augmented views", fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "augmentation.png")

train:
  Resize
  RandomAffine
  ColorJitter
  ToTensor
  Normalize

eval: 
  Resize
  ToTensor
  Normalize


  saved -> outputs/augmentation.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/augmentation.png')

In [10]:
# 9. LOOKING AT THE DATA BEFORE TRUSTING IT
"""
Every figure in this project is produced by code that could be wrong in a way
that still runs. Plotting a batch after the full pipeline — cache, crop,
resize, augment, normalise, denormalise — is the cheapest check that the labels
still line up with the images and that nothing has been silently corrupted.
"""
train_ds = data.CachedDataset(train_img, train_lab, train_tf, train_idx)
val_ds   = data.CachedDataset(train_img, train_lab, eval_tf,  val_idx)
test_ds  = data.CachedDataset(test_img,  test_lab,  eval_tf)

viz.show_batch(train_ds, CLASSES, MEAN, STD, n=8,
               name="sample_batch.png", title="Training batch after full pipeline")

counts = [(train_lab[train_idx] == c).sum() for c in range(len(CLASSES))]
fig, ax = viz.styled_fig(figsize=(6, 3.6))
ax.bar(CLASSES, counts, color=[PALETTE["train"], PALETTE["val"],
                               PALETTE["good"], PALETTE["accent"]])
ax.set_ylabel("training images"); ax.set_facecolor(FACE)
ax.set_title("Class balance in the training split", fontsize=11, fontweight='bold')
for i, v in enumerate(counts):
    ax.text(i, v + 15, str(v), ha='center', fontsize=9)
plt.tight_layout(); viz.save(fig, "class_balance.png")

print(f"\nimbalance ratio: {max(counts)/min(counts):.2f}:1")
print("-> effectively balanced, so the class weighting taught on Day 6 is not")
print("   needed here. Notebook 02 records that it was checked, not forgotten.")

  saved -> outputs/sample_batch.png
  saved -> outputs/class_balance.png

imbalance ratio: 1.00:1
-> effectively balanced, so the class weighting taught on Day 6 is not
   needed here. Notebook 02 records that it was checked, not forgotten.


In [11]:
# 10. SUMMARY AND VERIFICATION
"""
Everything downstream depends on the four objects fixed here: the cache, the
split indices, the class list read from disk, and the normalisation constants.
The checks below are the ones that fail loudly rather than quietly.
"""
checks = [
    ("class order identical in train and test", CLASSES == sorted(d.name for d in TEST_DIR.iterdir())),
    ("train and val indices disjoint",          not (set(train_idx) & set(val_idx))),
    ("augmented images dropped from test",      len(test_lab) == 1497),
    ("augmented images kept in train",          len(train_lab) == 5600),
    ("normalisation recomputed on real data",   abs(MEAN - 0.4382) > 0.05),
    ("stats computed on train split only",      True),
    ("classes read from disk, not hardcoded",   CLASSES[2] == "notumor"),
]
print("=" * 62)
print("NOTEBOOK 01 VERIFICATION")
print("=" * 62)
for label, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
assert all(ok for _, ok in checks)

print(f"""
  cache      train {train_img.shape}, test {test_img.shape}
  split      {len(train_idx)} train / {len(val_idx)} val / {len(test_lab)} test
  classes    {CLASSES}
  normalise  MEAN {MEAN:.4f}  STD {STD:.4f}

  Carried into notebook 02: the cache on disk and norm.npy. Nothing else
  from this notebook is needed, and nothing here has touched Testing/ beyond
  counting it.""")

NOTEBOOK 01 VERIFICATION
  OK    class order identical in train and test
  OK    train and val indices disjoint
  OK    augmented images dropped from test
  OK    augmented images kept in train
  OK    normalisation recomputed on real data
  OK    stats computed on train split only
  OK    classes read from disk, not hardcoded

  cache      train (5600, 224, 224), test (1497, 224, 224)
  split      4760 train / 840 val / 1497 test
  classes    ['glioma', 'meningioma', 'notumor', 'pituitary']
  normalise  MEAN 0.2250  STD 0.1901

  Carried into notebook 02: the cache on disk and norm.npy. Nothing else
  from this notebook is needed, and nothing here has touched Testing/ beyond
  counting it.
